# Project Three, Part One: Cozy Fantasy Dataset Cleaning

In [1]:
%pip install praw --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install pytrends --quiet

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import requests
import altair as alt
import praw
import pytrends

print("libraries ready")

libraries ready


In [4]:
from dotenv import load_dotenv
import os

load_dotenv()
API_KEY = os.getenv("GOOGLE_BOOKS_API_KEY")

In [5]:
#adding a time loop because I was getting 503 errors

import time

def search_google_books(title, author, max_retries=3):
    query = f"intitle:{title} inauthor:{author}"
    url = "https://www.googleapis.com/books/v1/volumes"
    params = {"q": query, "key": API_KEY}

    for attempt in range(max_retries):
        response = requests.get(url, params=params)
        data = response.json()
        if "error" not in data:
            return data
        if data["error"].get("code") == 503:
            time.sleep(2 * (attempt + 1)) 
            continue
        return data  

    return data  

In [6]:
result = search_google_books("Legends & Lattes", "Travis Baldree")
result

{'kind': 'books#volumes',
 'totalItems': 300,
 'items': [{'kind': 'books#volume',
   'id': '7lJzEAAAQBAJ',
   'etag': '9WzB4PW3gME',
   'selfLink': 'https://www.googleapis.com/books/v1/volumes/7lJzEAAAQBAJ',
   'volumeInfo': {'title': 'Legends & Lattes',
    'subtitle': 'A Novel of High Fantasy and Low Stakes',
    'authors': ['Travis Baldree'],
    'publisher': 'Tor Books',
    'publishedDate': '2022-06-07',
    'description': "An Instant New York Times Bestseller A Barnes & Noble Best Book of 2022 A Library Reads Pick An Indie Next Pick A Goodreads Best Fantasy Choice Award Nominee The much-beloved BookTok sensation, Travis Baldree's novel of high fantasy and low stakes. *This new edition includes a very special, never-before-seen bonus story, 'Pages to Fill.'* After a lifetime of bounties and bloodshed, Viv is hanging up her sword for the last time. The battle-weary orc aims to start fresh, opening the first ever coffee shop in the city of Thune. But old and new rivals stand in the 

In [7]:
# better info on the book out of the dictionary 

first_match = result["items"][0]
info = first_match["volumeInfo"]
print(info.get("title"))
print(info.get("authors"))
print(info.get("publisher"))
print(info.get("publishedDate"))
print(info.get("description"))
print(info.get("imageLinks", {}).get("thumbnail"))


Legends & Lattes
['Travis Baldree']
Tor Books
2022-06-07
An Instant New York Times Bestseller A Barnes & Noble Best Book of 2022 A Library Reads Pick An Indie Next Pick A Goodreads Best Fantasy Choice Award Nominee The much-beloved BookTok sensation, Travis Baldree's novel of high fantasy and low stakes. *This new edition includes a very special, never-before-seen bonus story, 'Pages to Fill.'* After a lifetime of bounties and bloodshed, Viv is hanging up her sword for the last time. The battle-weary orc aims to start fresh, opening the first ever coffee shop in the city of Thune. But old and new rivals stand in the way of success — not to mention the fact that no one has the faintest idea what coffee actually is. If Viv wants to put the blade behind her and make her plans a reality, she won't be able to go it alone. But the true rewards of the uncharted path are the travelers you meet along the way. And whether drawn together by ancient magic, flaky pastry, or a freshly brewed cup, th

In [8]:
#Review Open Library instead
def search_open_library(title, author):
    url = "https://openlibrary.org/search.json"
    params = {"title": title, "author": author}
    response = requests.get(url, params=params)
    return response.json()

ol_result = search_open_library("Legends & Lattes", "Travis Baldree")
ol_result["docs"][0] if ol_result["docs"] else "No match"


{'author_key': ['OL9109447A'],
 'author_name': ['Travis Baldree'],
 'cover_edition_key': 'OL38237074M',
 'cover_i': 13028635,
 'ebook_access': 'no_ebook',
 'edition_count': 19,
 'first_publish_year': 2022,
 'has_fulltext': False,
 'key': '/works/OL27591348W',
 'language': ['dut', 'spa', 'eng', 'ger'],
 'public_scan_b': False,
 'title': 'Legends & Lattes'}

## Load in manual cozy fantasy spreadsheet I created and add info with Google API

In [9]:
#load in csv

titles_df = pd.read_csv("../raw/raw_title_list.csv")
titles_df.head()


,title,author,approx_year,found_on_list,self_pub
0,Legends & Lattes,Travis Baldree,2022,Goodreads,True
1,The House in the Cerulean Sea,T.J. Klune,2020,Goodreads,False
2,The Spellshop,Sarah Beth Durst,2024,Goodreads,False
3,Emily Wilde's Encyclopaedia of Faeries,Heather Fawcett\r,2023,Goodreads,False
4,A Wizard’s Guide to Defensive Baking,T. Kingfisher,2020,Goodreads,False


In [11]:
#grabbing extra info and writing to a new clean csv
results = []

for idx, row in titles_df.iterrows():
    print(f"Processing {idx+1}/{len(titles_df)}: {row['title']}")
    try:
        data = search_google_books(row["title"], row["author"])
        ...
        if data.get("totalItems", 0) > 0:
            info = data["items"][0]["volumeInfo"]
            results.append({
                "title": row["title"],
                "author": row["author"],
                "publisher_raw": info.get("publisher"),
                "published_date": info.get("publishedDate"),
                "page_count": info.get("pageCount"),
                "description": info.get("description"),
                "cover_url": info.get("imageLinks", {}).get("thumbnail"),
                "found_on_list": row["found_on_list"],
            })
        else:
            results.append({"title": row["title"], "author": row["author"], "publisher_raw": None})
    except Exception as e:
        print(f"Failed on {row['title']}: {e}")
        results.append({"title": row["title"], "author": row["author"], "publisher_raw": None})

    time.sleep(1)  

metadata_df = pd.DataFrame(results)
metadata_df.to_csv("../raw/google_books_metadata.csv", index=False)



Processing 1/99: Legends & Lattes
Processing 2/99: The House in the Cerulean Sea
Processing 3/99: The Spellshop
Processing 4/99: Emily Wilde's Encyclopaedia of Faeries
Processing 5/99: A Wizard’s Guide to Defensive Baking
Processing 6/99: The Very Secret Society of Irregular Witches
Processing 7/99: A Psalm for the Wild-Built
Processing 8/99: Bookshops & Bonedust
Processing 9/99: Can't Spell Treason Without Tea
Processing 10/99: Tress of the Emerald Sea
Processing 11/99: The Teller of Small Fortunes
Processing 12/99: Brigands & Breadknives
Processing 13/99: A Witch's Guide to Magical Innkeeping
Processing 14/99: Emily Wilde's Compendium of Lost Tales
Processing 15/99: Nettle & Bone
Processing 16/99: Half a Soul
Processing 17/99: Agnes Aubert's Mystical Cat Shelter
Processing 18/99: A Coup of Tea
Processing 19/99: The House Witch
Processing 20/99: Howl's Moving Castle
Processing 21/99: A Fellowship of Bakers & Magic
Processing 22/99: That Time I Got Drunk and Saved a Demon
Processing 23

In [12]:
metadata_df[metadata_df["publisher_raw"].isna()]

,title,author,publisher_raw,published_date,page_count,description,cover_url,found_on_list
4,A Wizard’s Guide to Defensive Baking,T. Kingfisher,NaN,2020-07-21,NaN,Fourteen-year-old Mona isn't like the wizards ...,http://books.google.com/books/content?id=ZiWbz...,Goodreads
5,The Very Secret Society of Irregular Witches,Sangu Mandanna\r\n,NaN,NaN,NaN,NaN,NaN,NaN
15,Half a Soul,Olivia Atwater\r,NaN,NaN,NaN,NaN,NaN,NaN
17,A Coup of Tea,Casey Blair,NaN,2022,0.0,"""When the fourth princess of Istalam is due to...",NaN,Reddit
21,That Time I Got Drunk and Saved a Demon,Kimberly Lemming,NaN,NaN,NaN,NaN,NaN,NaN
22,The Tea Dragon Society,K. O'Neill,NaN,NaN,NaN,NaN,NaN,NaN
25,Cursed Cocktails,S.L. Rowland,NaN,NaN,NaN,NaN,NaN,NaN
27,Violet Thistlewaite Is Not a Villain Anymore,Emily Krempholtz\r,NaN,NaN,NaN,NaN,NaN,NaN
29,A Pirate's Life for Tea,Rebecca Thorne,NaN,2023-02-20,0.0,"While hunting for missing dragon eggs, Kianthe...",http://books.google.com/books/content?id=VOScz...,Goodreads
31,Keeper of Enchanted Rooms,Charlie N. Holmberg,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
import re

def clean_text(text):
    if pd.isna(text):
        return text
    cleaned = re.sub(r'[\r\n]+', ' ', text)   # replace any carriage returns/newlines with a space
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()  # collapse leftover double-spaces, trim ends
    return cleaned

titles_df["title"] = titles_df["title"].apply(clean_text)
titles_df["author"] = titles_df["author"].apply(clean_text)

In [14]:
missing_mask = metadata_df["publisher_raw"].isna()
missing_df = metadata_df[missing_mask].copy()
print(f"{len(missing_df)} titles missing data — retrying with cleaned text")

# pull the now-cleaned title/author back in for these same rows
missing_df["title"] = titles_df.loc[missing_df.index, "title"]
missing_df["author"] = titles_df.loc[missing_df.index, "author"]

for idx, row in missing_df.iterrows():
    print(f"Retrying: {row['title']}")
    data = search_google_books(row["title"], row["author"])
    if data.get("totalItems", 0) > 0:
        info = data["items"][0]["volumeInfo"]
        metadata_df.loc[idx, "publisher_raw"] = info.get("publisher")
        metadata_df.loc[idx, "published_date"] = info.get("publishedDate")
        metadata_df.loc[idx, "page_count"] = info.get("pageCount")
        metadata_df.loc[idx, "description"] = info.get("description")
        metadata_df.loc[idx, "cover_url"] = info.get("imageLinks", {}).get("thumbnail")
    time.sleep(1)

51 titles missing data — retrying with cleaned text
Retrying: A Wizard’s Guide to Defensive Baking
Retrying: The Very Secret Society of Irregular Witches
Retrying: Half a Soul
Retrying: A Coup of Tea
Retrying: That Time I Got Drunk and Saved a Demon
Retrying: The Tea Dragon Society
Retrying: Cursed Cocktails
Retrying: Violet Thistlewaite Is Not a Villain Anymore
Retrying: A Pirate's Life for Tea
Retrying: Keeper of Enchanted Rooms
Retrying: The Undertaking of Hart and Mercy
Retrying: The Keeper of Magical Things
Retrying: The Phoenix Keeper
Retrying: The Baby Dragon Café
Retrying: The Bookshop and the Barbarian
Retrying: Stay for a Spell
Retrying: Forged by Magic
Retrying: The Cybernetic Tea Shop
Retrying: The Long Way to a Small, Angry Planet
Retrying: Mindtouch
Retrying: Coffee, Milk & Spider Silk
Retrying: The Wizard's Butler
Retrying: House of Many Ways
Retrying: Vanessa Yu's Magical Paris Tea Shop
Retrying: Magician's Hoard
Retrying: Mr. Penumbra's 24-Hour Bookstore
Retrying: The 

In [16]:
def search_open_library(title, author, timeout=10):
    url = "https://openlibrary.org/search.json"
    params = {"title": title, "author": author}
    response = requests.get(url, params=params, timeout=timeout)
    return response.json()

still_missing_mask = metadata_df["publisher_raw"].isna()
still_missing_df = metadata_df[still_missing_mask].copy()
print(f"{len(still_missing_df)} titles heading to Open Library")

for idx, row in still_missing_df.iterrows():
    print(f"Open Library lookup: {row['title']}")
    try:
        data = search_open_library(row["title"], row["author"])
        if data.get("numFound", 0) > 0:
            doc = data["docs"][0]
            publishers = doc.get("publisher")
            metadata_df.loc[idx, "publisher_raw"] = publishers[0] if publishers else None

            year = doc.get("first_publish_year")
            metadata_df.loc[idx, "published_date"] = str(year) if year is not None else None

            metadata_df.loc[idx, "page_count"] = doc.get("number_of_pages_median")
            cover_id = doc.get("cover_i")
            metadata_df.loc[idx, "cover_url"] = f"https://covers.openlibrary.org/b/id/{cover_id}-M.jpg" if cover_id else None
            metadata_df.loc[idx, "source"] = "open_library"
    except Exception as e:
        print(f"Failed on {row['title']}: {e}")
    time.sleep(1)

37 titles heading to Open Library
Open Library lookup: A Wizard’s Guide to Defensive Baking
Open Library lookup: A Coup of Tea
Open Library lookup: That Time I Got Drunk and Saved a Demon
Open Library lookup: Violet Thistlewaite Is Not a Villain Anymore
Open Library lookup: A Pirate's Life for Tea
Open Library lookup: The Undertaking of Hart and Mercy
Open Library lookup: The Keeper of Magical Things
Open Library lookup: The Baby Dragon Café
Open Library lookup: The Bookshop and the Barbarian
Open Library lookup: Forged by Magic
Open Library lookup: The Cybernetic Tea Shop
Open Library lookup: Mindtouch
Open Library lookup: The Wizard's Butler
Open Library lookup: House of Many Ways
Open Library lookup: Magician's Hoard
Open Library lookup: Mr. Penumbra's 24-Hour Bookstore
Open Library lookup: The Herbwitch's Apprentice
Open Library lookup: The Witching Flour
Open Library lookup: Fanuilh
Open Library lookup: Calculated Whisk
Open Library lookup: Drop of a Hat
Open Library lookup: Tempe

In [18]:
metadata_df.to_csv("../raw/google_books_metadata.csv", index=False)
truly_missing = metadata_df[metadata_df["publisher_raw"].isna()]
print(f"{len(truly_missing)} titles not found in either source")
truly_missing[["title", "author"]]

37 titles not found in either source


,title,author
4,A Wizard’s Guide to Defensive Baking,T. Kingfisher
17,A Coup of Tea,Casey Blair
21,That Time I Got Drunk and Saved a Demon,Kimberly Lemming
27,Violet Thistlewaite Is Not a Villain Anymore,Emily Krempholtz\r
29,A Pirate's Life for Tea,Rebecca Thorne
33,The Undertaking of Hart and Mercy,Megan Bannen
37,The Keeper of Magical Things,Julie Leong
39,The Baby Dragon Café,Aamna Qureshi
40,The Bookshop and the Barbarian,Morgan Stang
43,Forged by Magic,Jenna Wolfhart\r


In [19]:
metadata_df = metadata_df.drop_duplicates(subset=["title", "author"])
metadata_df["published_date"] = pd.to_datetime(metadata_df["published_date"], errors="coerce")
metadata_df["pub_year"] = metadata_df["published_date"].dt.year


In [20]:
traditional_imprints = [
    "tor", "bramble", "ace", "orbit", "del rey", "podium",
    "harpervoyager", "angry robot", "daw", "gollancz", "saga press"
]

def classify_publisher(publisher_raw):
    if pd.isna(publisher_raw):
        return "unknown"
    p = publisher_raw.lower()
    if "independently published" in p:
        return "self-published"
    for imprint in traditional_imprints:
        if imprint in p:
            return "traditional"
    return "likely self-published"  # catches small/unrecognized presses too — flag for spot-check

metadata_df["pub_category"] = metadata_df["publisher_raw"].apply(classify_publisher)
metadata_df["pub_category"].value_counts()


pub_category
likely self-published    42
unknown                  37
traditional              19
self-published            1
Name: count, dtype: int64

In [21]:
metadata_df[metadata_df["pub_category"] == "likely self-published"][["title", "author", "publisher_raw"]]

,title,author,publisher_raw
3,Emily Wilde's Encyclopaedia of Faeries,Heather Fawcett\r,Random House
5,The Very Secret Society of Irregular Witches,Sangu Mandanna\r\n,Penguin
10,The Teller of Small Fortunes,Julie Leong,Penguin
12,A Witch's Guide to Magical Innkeeping,Sangu Mandanna\n,Penguin
19,Howl's Moving Castle,Diana Wynne Jones,HarperCollins Children's Books
20,A Fellowship of Bakers & Magic,J. Penner,Poisoned Pen Press
22,The Tea Dragon Society,K. O'Neill,Random House
24,The Honey Witch,Sydney J. Shields\r,Redhook
25,Cursed Cocktails,S.L. Rowland,Tales of Aedrea
28,Rewitched,Lucy Jane Wood,Penguin


In [22]:
uncertain = metadata_df[metadata_df["pub_category"] == "likely self-published"]
publisher_counts = uncertain["publisher_raw"].value_counts()
publisher_counts

publisher_raw
Penguin                                  10
Random House                              5
47North                                   2
HarperCollins                             2
Underhill Books                           2
Simon and Schuster                        2
HarperCollins Children's Books            1
Poisoned Pen Press                        1
Redhook                                   1
Tales of Aedrea                           1
Hachette UK                               1
World Tree Publishing                     1
Hodderscape                               1
Ridan Publishing                          1
Regency Dragons                           1
Coyote JM Edwards                         1
Kim Watt                                  1
Harperfire                                1
Houghton Mifflin Harcourt                 1
Bloomsbury Publishing                     1
Ballantine Books                          1
Tourmaline & Quartz Publishing LLC        1
Little, Brown Book

In [24]:
manual_corrections = {
    "Penguin": "traditional",
    "Random House": "traditional",
    "47North": "traditional",  
    "HarperCollins": "traditional",
    "Simon and Schuster": "traditional",
    "HarperCollins Children's Books": "traditional",
    "Poisoned Pen Press": "traditional",
    "Redhook": "traditional",
    "Hachette UK": "traditional",
    "Hodderscape": "traditional",
    "Houghton Mifflin Harcourt": "traditional",
    "Bloomsbury Publishing": "traditional",
    "Ballantine Books": "traditional",
    "Little, Brown Books for Young Readers": "traditional",
    "Headline": "traditional",
    "Macmillan": "traditional",
    "Ridan Publishing": "traditional",

    # Personal/author-run imprints — functionally self-published
    "Tales of Aedrea": "self-published",
    "Regency Dragons": "self-published",
    "Coyote JM Edwards": "self-published",
    "Kim Watt": "self-published",
    "Harperfire": "self-published",
    "Tourmaline & Quartz Publishing LLC": "self-published",
    "Underhill Books": "self-published",
    "World Tree Publishing": "self-published"
}

def apply_manual_override(row):
    if row["publisher_raw"] in manual_corrections:
        return manual_corrections[row["publisher_raw"]]
    return row["pub_category"]

metadata_df["pub_category"] = metadata_df.apply(apply_manual_override, axis=1)
metadata_df["pub_category"].value_counts()

pub_category
traditional       52
unknown           37
self-published    10
Name: count, dtype: int64

In [27]:
truly_missing = metadata_df[metadata_df["publisher_raw"].isna()]
print(f"{len(truly_missing)} titles still missing")
truly_missing[["title", "author"]]

37 titles still missing


,title,author
4,A Wizard’s Guide to Defensive Baking,T. Kingfisher
17,A Coup of Tea,Casey Blair
21,That Time I Got Drunk and Saved a Demon,Kimberly Lemming
27,Violet Thistlewaite Is Not a Villain Anymore,Emily Krempholtz\r
29,A Pirate's Life for Tea,Rebecca Thorne
33,The Undertaking of Hart and Mercy,Megan Bannen
37,The Keeper of Magical Things,Julie Leong
39,The Baby Dragon Café,Aamna Qureshi
40,The Bookshop and the Barbarian,Morgan Stang
43,Forged by Magic,Jenna Wolfhart\r


In [28]:
import re

def simplify_title(title):
    simplified = re.sub(r"\(.*?\)", "", title)   # drop parenthetical series info
    simplified = simplified.split(":")[0]          # drop anything after a colon
    return simplified.strip()

for idx, row in truly_missing.iterrows():
    clean_title = simplify_title(row["title"])
    print(f"Trying: '{clean_title}' (original: '{row['title']}')")

Trying: 'A Wizard’s Guide to Defensive Baking' (original: 'A Wizard’s Guide to Defensive Baking')
Trying: 'A Coup of Tea' (original: 'A Coup of Tea')
Trying: 'That Time I Got Drunk and Saved a Demon' (original: 'That Time I Got Drunk and Saved a Demon')
Trying: 'Violet Thistlewaite Is Not a Villain Anymore' (original: 'Violet Thistlewaite Is Not a Villain Anymore')
Trying: 'A Pirate's Life for Tea' (original: 'A Pirate's Life for Tea')
Trying: 'The Undertaking of Hart and Mercy' (original: 'The Undertaking of Hart and Mercy')
Trying: 'The Keeper of Magical Things' (original: 'The Keeper of Magical Things')
Trying: 'The Baby Dragon Café' (original: 'The Baby Dragon Café')
Trying: 'The Bookshop and the Barbarian' (original: 'The Bookshop and the Barbarian')
Trying: 'Forged by Magic' (original: 'Forged by Magic')
Trying: 'The Cybernetic Tea Shop' (original: 'The Cybernetic Tea Shop')
Trying: 'Mindtouch' (original: 'Mindtouch')
Trying: 'The Wizard's Butler' (original: 'The Wizard's Butler'

In [29]:
def search_google_books_loose(title, timeout=10):
    url = "https://www.googleapis.com/books/v1/volumes"
    params = {"q": title, "key": API_KEY}
    response = requests.get(url, params=params, timeout=timeout)
    return response.json()

for idx, row in truly_missing.iterrows():
    clean_title = simplify_title(row["title"])
    data = search_google_books_loose(clean_title)
    if data.get("totalItems", 0) > 0:
        info = data["items"][0]["volumeInfo"]
        print(f"Original: {row['title']} | Author: {row['author']}")
        print(f"  → Found: {info.get('title')} | Publisher: {info.get('publisher')}")
        print("---")
    else:
        print(f"Still no match: {row['title']}")
    time.sleep(1)

Original: A Wizard’s Guide to Defensive Baking | Author: T. Kingfisher
  → Found: A Wizard's Guide to Defensive Baking | Publisher: None
---
Original: A Coup of Tea | Author: Casey Blair
  → Found: A Coup of Tea | Publisher: None
---
Original: That Time I Got Drunk and Saved a Demon | Author: Kimberly Lemming
  → Found: That Time I Got Drunk And Saved A Demon | Publisher: None
---
Original: Violet Thistlewaite Is Not a Villain Anymore | Author: Emily Krempholtz
  → Found: Violet Thistlewaite Is Not a Villain Anymore | Publisher: Penguin
---
Still no match: A Pirate's Life for Tea
Original: The Undertaking of Hart and Mercy | Author: Megan Bannen
  → Found: The Undertaking of Hart and Mercy | Publisher: Orbit
---
Still no match: The Keeper of Magical Things
Original: The Baby Dragon Café | Author: Aamna Qureshi
  → Found: The Baby Dragon Cafe (The Baby Dragon series) | Publisher: HarperCollins UK
---
Original: The Bookshop and the Barbarian | Author: Morgan Stang
  → Found: The Bookshop

In [30]:
still_missing = metadata_df[metadata_df["publisher_raw"].isna()]
print(len(still_missing))
still_missing[["title", "author"]]

37


,title,author
4,A Wizard’s Guide to Defensive Baking,T. Kingfisher
17,A Coup of Tea,Casey Blair
21,That Time I Got Drunk and Saved a Demon,Kimberly Lemming
27,Violet Thistlewaite Is Not a Villain Anymore,Emily Krempholtz\r
29,A Pirate's Life for Tea,Rebecca Thorne
33,The Undertaking of Hart and Mercy,Megan Bannen
37,The Keeper of Magical Things,Julie Leong
39,The Baby Dragon Café,Aamna Qureshi
40,The Bookshop and the Barbarian,Morgan Stang
43,Forged by Magic,Jenna Wolfhart\r


In [31]:
still_missing_now = metadata_df[metadata_df["publisher_raw"].isna()]
print(len(still_missing_now))
still_missing_now[["title", "author"]]

37


,title,author
4,A Wizard’s Guide to Defensive Baking,T. Kingfisher
17,A Coup of Tea,Casey Blair
21,That Time I Got Drunk and Saved a Demon,Kimberly Lemming
27,Violet Thistlewaite Is Not a Villain Anymore,Emily Krempholtz\r
29,A Pirate's Life for Tea,Rebecca Thorne
33,The Undertaking of Hart and Mercy,Megan Bannen
37,The Keeper of Magical Things,Julie Leong
39,The Baby Dragon Café,Aamna Qureshi
40,The Bookshop and the Barbarian,Morgan Stang
43,Forged by Magic,Jenna Wolfhart\r


In [32]:
metadata_df.to_csv("../clean/cozy_fantasy_corpus.csv", index=False)

In [35]:
#re-upload manually completed dataset
metadata_df = pd.read_csv("../raw/raw_title_list - cozy_fantasy_pub_cats.csv")
metadata_df["pub_category"].value_counts()

pub_category
traditional       64
self-published    35
Name: count, dtype: int64